# Fuel Lattice Parameter: Prediction

Predicts lattice parameters for known experimental compositions and compares against
their measured values, using `engine.predict.predict_csv`, the exact code path behind
`cli.py predict --csv` and the `predict_lattice_parameter` MCP tool.

All four reportable models (`rf1`, `rf2`, `gbr1`, `gbr2`) predict every row; each gets
its own `a_pred_<key>` column and its own MAE/MSE against `a_true`.

**Benchmark inputs live in `Data/benchmarks/`:**
- `UNUC.csv`, the U(N,C) system (23 rows), values based on an experimental fit
- `CeO2Nd2O3Vals.csv`, the (Ce,Nd)O2 system (7 rows)

**The `ref_mp-id` column** names the reference *host structure* whose symmetry is fed to
the model, so an input and not a label. One rule governs it, the same one
`engine/reference_resolver.py` applies when the column is absent: **the most prevalent
end-member's mp-id; for a 50/50 mix, the more stable one's.** So `UNUC.csv` uses UN
(`mp-1865`) where `y > 0.5` and UC (`mp-2489`) where `y < 0.5`; the `y = 0.5` row ties on
`energy_above_hull` (both 0.0, since each is a line compound on its own chemsys hull) and is
decided by formation energy, where UN (−1.582 eV/atom) beats UC (−0.255). `CeO2Nd2O3Vals.csv`
is Ce-dominant on every row, so CeO2 (`mp-20194`) throughout.

To benchmark another system, add a CSV with `composition, ref_mp-id, a_true` columns to
`Data/benchmarks/` and point a cell below at it.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from engine import config
from engine.predict import predict_csv

## U(N,C) system

In [ ]:
result = predict_csv(str(config.DATA_DIR / "benchmarks" / "UNUC.csv"))
print(f"{result['rows']} rows predicted")
for w in result['warnings']:
    print(f"  ! {w}")
df_unc = result['dataframe']
display(df_unc)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 9))

x = df_unc['composition'].str.extract(r'C([\d.]+)').astype(float).fillna(0.0).iloc[:, 0]
ax.scatter(x, df_unc['a_true'], marker='o', s=90, color='red', label='True $a$')

styles = {
    'rf1':  ('s', 'orange',    config.MODEL_FILES['rf1'][1]),
    'rf2':  ('^', 'tab:green', config.MODEL_FILES['rf2'][1]),
    'gbr1': ('D', 'tab:blue',  config.MODEL_FILES['gbr1'][1]),
    'gbr2': ('v', 'tab:purple', config.MODEL_FILES['gbr2'][1]),
}
for key, (marker, color, name) in styles.items():
    col = f'a_pred_{key}'
    if col in df_unc.columns:
        mae = result['metrics'].get(key, {}).get('mae')
        label = f'{name} ({key})' + (f' — MAE {mae:.4f} Å' if mae is not None else '')
        ax.scatter(x, df_unc[col], marker=marker, color=color, alpha=0.8, label=label)

ax.set_xlabel('C content $x$ in UN$_{1-x}$C$_x$', fontsize=16, fontweight='bold')
ax.set_ylabel('Lattice Parameter $a$ (Å)', fontsize=16, fontweight='bold')
ax.set_title('True vs Predicted Lattice Parameter of UN$_{1-x}$C$_x$', fontsize=18, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## (Ce,Nd)O2 system

In [ ]:
result_ceo2 = predict_csv(str(config.DATA_DIR / "benchmarks" / "CeO2Nd2O3Vals.csv"))
print(f"{result_ceo2['rows']} rows predicted")
for w in result_ceo2['warnings']:
    print(f"  ! {w}")
df_ceo2 = result_ceo2['dataframe']
display(df_ceo2)

## Feature correlation with lattice parameter (cubic systems)

Spearman correlation between each training feature and the true lattice parameter `a`,
restricted to cubic-system rows, for features with |correlation| > 0.2. This is
descriptive analysis of the training data, not part of the prediction pipeline, and
`Dataset/Training_Dataset.csv` must exist locally (run the model-creation notebook, or
`cli.py train`, first).

The plot below is rendered but not saved. `python scripts/feature_correlations.py` writes
the same correlations to `Results/metrics/feature_spearman_cubic.csv`, covering all 145
features rather than only those above the 0.2 display threshold, which is what the
published figure's underlying values are taken from.

In [ ]:
import seaborn as sns
import joblib

if config.DATASET_TRAINING.exists():
    df_train = pd.read_csv(config.DATASET_TRAINING, low_memory=False)
    df_train = df_train[df_train['crystal_system'] == 'cubic']
    feature_labels = joblib.load(config.ML_FEATURELABELS)

    correlations = df_train[['a'] + feature_labels].corr(method='spearman')
    a_corr = correlations['a'].drop('a')

    corr_df = a_corr.reset_index()
    corr_df.columns = ['Feature', 'Correlation']
    corr_df['AbsCorr'] = corr_df['Correlation'].abs()
    filtered_corr = corr_df.sort_values(by='AbsCorr', ascending=False)
    filtered_corr = filtered_corr[filtered_corr['AbsCorr'] > 0.2]

    plt.figure(figsize=(12, 12))
    sns.barplot(data=filtered_corr, x='Correlation', y='Feature', palette='coolwarm')
    plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
    plt.xlabel('Spearman Correlation with Lattice Parameter', fontsize=16)
    plt.ylabel('Input Feature Label', fontsize=16)
    plt.yticks(fontsize=11, rotation=20)
    plt.xticks(fontsize=14)
    plt.title(
        "Spearman Correlation of Features for Lattice Parameter\n"
        "of Cubic Crystal Systems (>= 0.2 or <= -0.2)", fontsize=20
    )
    plt.tight_layout()
    plt.show()
    display(filtered_corr)
else:
    print(f"{config.DATASET_TRAINING} not found — run the model-creation "
          "notebook or `cli.py train` first to generate it.")